- R²
- Loss curves (both models on one plot)
- Predicted vs actual scatter plots
- Written conclusion

In [ ]:
#Imports to run models
import sklearn
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import r2_score
import pandas as pd

import sklearn
from sklearn.metrics import mean_squared_error
from sklearn import preprocessing
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor

In [ ]:
from preprocessing import df_basic_cleaning_and_split, array_standardise_data


In [ ]:
# from model1_baseline.ipynb
def set_seed(seed=0):
    np.random.seed(seed)

def relu(z):
    return np.maximum(0, z)

def drelu(a):
    return (a > 0).astype(float)

def mse_loss(y_pred, y_true, n):
    return 0.5 * np.mean((y_pred - y_true) ** 2)

num_hidden_neurons = 64

def forward(X, W1, b1, w2, b2):
    z1 = X @ W1 + b1
    a1 = relu(z1)
    z2 = a1 @ w2 + b2
    y_hat = z2
    cache = {"z1": z1, "a1": a1, "z2": z2, "y_hat": y_hat}
    return y_hat, cache

def backprop(X, y, W1, b1, w2, b2):
    N = X.shape[0]
    y_hat, cache = forward(X, W1, b1, w2, b2)
    a1 = cache["a1"]
    dL_dyhat = (y_hat - y) / N
    dL_dz2 = dL_dyhat
    dw2 = a1.T @ dL_dz2
    db2 = np.sum(dL_dz2, axis=0)
    dL_da1 = dL_dz2 @ w2.T
    dL_dz1 = dL_da1 * drelu(a1)
    dW1 = X.T @ dL_dz1
    db1 = np.sum(dL_dz1, axis=0)
    return {"dW1": dW1, "db1": db1, "dw2": dw2, "db2": db2}

def train_NN(X, y, hidden=num_hidden_neurons, lr=1.0, iters=4000, seed=2):
    set_seed(seed)
    W1 = 0.5 * np.random.randn(X.shape[1], hidden)
    b1 = np.zeros(hidden)
    w2 = 0.5 * np.random.randn(hidden, 1)
    b2 = np.zeros(1)
    losses = []
    for t in range(iters):
        y_hat, _ = forward(X, W1, b1, w2, b2)
        L = mse_loss(y_hat, y, n=X.shape[0])
        losses.append(L)
        grads = backprop(X, y, W1, b1, w2, b2)
        W1 -= lr * grads["dW1"]
        b1 -= lr * grads["db1"]
        w2 -= lr * grads["dw2"]
        b2 -= lr * grads["db2"]
    return W1, b1, w2, b2, losses

def predict(X, W1, b1, w2, b2):
    y_hat, _ = forward(X, W1, b1, w2, b2)
    return y_hat